# Observatorio Nacional del Mercado Eléctrico Colombiano
## Entrega 2 — Extracción automatizada e integración en base maestra

**Introducción a la Analítica de Negocios · Universidad de Antioquia**
**Integrantes:** Juan Esteban Diosa Sanmartín · Juliana Hurtado Mosquera · Daniel Alejandro Osorio

**Objetivo:** analizar día a día la relación entre la demanda, el precio de bolsa y la disponibilidad hídrica de energía en Colombia, y cómo el calendario y el contexto macroeconómico inciden en esas variaciones. Grano nacional-diario.

**Preguntas analíticas:**
1. ¿Cómo se relaciona el nivel de embalses (disponibilidad hídrica) con el precio de bolsa nacional de energía?
2. ¿Cómo varía la demanda nacional de energía entre días hábiles, fines de semana y festivos?
3. ¿Cómo se relaciona el precio de bolsa con la tasa de cambio (TRM) y con el IPC de energía?

**Fuentes de datos:** XM/SIMEM (demanda, precio, embalses), TRM (Superfinanciera/Banco de la República), IPC (DANE). El calendario de festivos es una variable auxiliar, no una fuente externa.


## 1. Configuración
Librerías, parámetros de fecha a extraer y carpetas de salida.

In [11]:
import os
import re
import io
import time
import unicodedata
import datetime as dt

import requests
import numpy as np
import pandas as pd

FECHA_INICIO = "2024-01-01"
FECHA_FIN    = "2024-03-31"

RUTA_CRUDOS     = "datos_crudos"
RUTA_PROCESADOS = "datos_procesados"
os.makedirs(RUTA_CRUDOS, exist_ok=True)
os.makedirs(RUTA_PROCESADOS, exist_ok=True)

HEADERS_HTTP = {"User-Agent": "Mozilla/5.0 (compatible; ObservatorioEnergiaCO/1.0; +academico UdeA)"}


Repite una llamada hasta 3 veces si falla por un error de red/tiempo de
espera; si el error es de otro tipo, lo deja pasar de inmediato.

In [12]:
def solicitar_con_reintentos(funcion, intentos=3, espera_seg=8):
    """Ejecuta `funcion` (sin argumentos) hasta `intentos` veces si ocurre
    un error de conexión o de tiempo de espera. Si el error es de otro tipo
    (por ejemplo, un dato no encontrado), se deja pasar de inmediato porque
    reintentarlo no serviría de nada."""
    for intento in range(1, intentos + 1):
        try:
            return funcion()
        except (requests.exceptions.RequestException, TimeoutError, ConnectionError) as error:
            print(f"  intento {intento}/{intentos} falló por un error de red: {error}")
            if intento == intentos:
                raise
            time.sleep(espera_seg)


## 2. XM / SIMEM — demanda, precio de bolsa y disponibilidad hídrica (`Entity="Sistema"`)

Conecta con la API de XM (instalando la librería si falta), y guarda el
cliente y el catálogo de métricas en caché para no volver a descargarlos.

In [13]:
try:
    from pydataxm.pydataxm import ReadDB
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "pydataxm", "--break-system-packages", "-q"])
    from pydataxm.pydataxm import ReadDB

# El catálogo de métricas de XM se descarga una sola vez y se reutiliza.
_cliente_xm = None
_catalogo_xm = None


def obtener_cliente_y_catalogo():
    global _cliente_xm, _catalogo_xm
    if _cliente_xm is None:
        _cliente_xm = solicitar_con_reintentos(lambda: ReadDB())
        _catalogo_xm = _cliente_xm.get_collections()
        print(f"Catálogo de XM descargado: {len(_catalogo_xm)} métricas.")
    return _cliente_xm, _catalogo_xm


def buscar_metrica(catalogo, texto, entidad):
    """Recorre las columnas de texto del catálogo buscando `texto` (sin
    importar mayúsculas/tildes), para la `entidad` indicada. Devuelve el
    MetricId de la primera coincidencia, o None si no encuentra nada."""
    subset = catalogo[catalogo["Entity"] == entidad]
    texto = texto.lower()
    for columna in subset.columns:
        if columna in ("MetricId", "Entity"):
            continue
        coincidencias = subset[subset[columna].astype(str).str.lower().str.contains(texto, na=False)]
        if not coincidencias.empty:
            return coincidencias.iloc[0]["MetricId"]
    return None


Agrega la respuesta de XM a un valor diario, extrae demanda/precio a
nivel nacional, y usa el volumen de embalses como proxy de generación.

In [14]:
def valores_diarios(df, agregacion):
    """Convierte la respuesta de XM (columnas horarias Values_Hour01..24, o
    una sola columna diaria) en una serie diaria numérica."""
    columnas_horas = [c for c in df.columns if c.startswith("Values_Hour")]
    if columnas_horas:
        for c in columnas_horas:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        if agregacion == "suma":
            df["valor_dia"] = df[columnas_horas].sum(axis=1)
        else:
            df["valor_dia"] = df[columnas_horas].mean(axis=1)
    else:
        columna_valor = "Values_Value" if "Values_Value" in df.columns else "Value"
        df["valor_dia"] = pd.to_numeric(df[columna_valor], errors="coerce")

    df["fecha"] = pd.to_datetime(df["Date"]).dt.date
    return df


def extraer_metrica_sistema(nombres_posibles, nombre_columna, agregacion, fecha_inicio, fecha_fin, factor=1.0):
    """Extrae una métrica de XM a nivel nacional (Entity='Sistema').
    `nombres_posibles` puede ser un texto o una lista de textos candidatos
    (se usa el primero que exista en el catálogo real)."""
    cliente, catalogo = obtener_cliente_y_catalogo()
    if isinstance(nombres_posibles, str):
        nombres_posibles = [nombres_posibles]

    metric_id = None
    for nombre in nombres_posibles:
        metric_id = buscar_metrica(catalogo, nombre, "Sistema")
        if metric_id:
            break
    if metric_id is None:
        raise RuntimeError(f"No se encontró ninguna métrica para {nombres_posibles}.")

    df = solicitar_con_reintentos(lambda: cliente.request_data(
        metric_id, "Sistema", dt.date.fromisoformat(fecha_inicio), dt.date.fromisoformat(fecha_fin)))
    if df.empty:
        raise RuntimeError(f"La métrica '{metric_id}' no devolvió datos.")

    df = valores_diarios(df, agregacion)
    salida = df.groupby("fecha", as_index=False)["valor_dia"].agg(agregacion if agregacion == "sum" else "mean")
    salida[nombre_columna] = salida["valor_dia"] * factor
    return salida[["fecha", nombre_columna]]


def extraer_volumen_embalses(fecha_inicio, fecha_fin):
    """Volumen útil de los embalses del SIN: suma agregada, a nivel
    nacional, de la energía almacenada en todos los embalses del país
    (Entity='Sistema') — proxy de disponibilidad hídrica."""
    return extraer_metrica_sistema("Volumen Útil", "volumen_util_embalses_kwh", "promedio", fecha_inicio, fecha_fin)


## 3. TRM — Tasa Representativa del Mercado (Superintendencia Financiera / Banco de la República)

TRM histórica desde el dataset oficial de datos.gov.co (mcec-87by).

In [15]:
def extraer_trm(fecha_inicio, fecha_fin):
    """TRM histórica, dataset oficial `mcec-87by` de datos.gov.co."""
    url = "https://www.datos.gov.co/resource/mcec-87by.json"

    # Se descubre el nombre real de las columnas con una muestra pequeña,
    # en vez de asumirlo, por si el portal cambia los nombres de campo.
    muestra = solicitar_con_reintentos(
        lambda: requests.get(url, params={"$limit": 1}, headers=HEADERS_HTTP, timeout=20).json())
    columnas = list(muestra[0].keys())
    columna_fecha = next(c for c in columnas if "vigencia" in c.lower() or "fecha" in c.lower())
    columna_valor = next(c for c in columnas if "valor" in c.lower())

    params = {
        "$where": f"{columna_fecha} between '{fecha_inicio}T00:00:00' and '{fecha_fin}T23:59:59'",
        "$limit": 5000,
        "$order": columna_fecha,
    }
    datos = solicitar_con_reintentos(lambda: requests.get(url, params=params, headers=HEADERS_HTTP, timeout=30).json())
    df = pd.DataFrame(datos)
    if df.empty:
        raise RuntimeError(f"La TRM no devolvió registros entre {fecha_inicio} y {fecha_fin}.")

    df["fecha"] = pd.to_datetime(df[columna_fecha]).dt.date
    df["trm"] = pd.to_numeric(df[columna_valor], errors="coerce")
    return df.groupby("fecha", as_index=False)["trm"].mean()


## 4. IPC — Índice de Precios al Consumidor de energía (descarga directa del DANE)

Se descarga el anexo mensual oficial del DANE y se ubica la fila de la división **"Alojamiento, Agua, Electricidad, Gas y Otros Combustibles"**. El DANE cambió la estructura de carpetas de estos archivos durante 2024 (algunos meses no usan subcarpeta), por lo que se prueban dos rutas posibles. Una vez ubicada la fila, se lee la columna de variación anual usando su posición estándar dentro del anexo (columna E, quinta columna desde la división).

In [16]:
MESES_DANE = {1:"ene",2:"feb",3:"mar",4:"abr",5:"may",6:"jun",7:"jul",8:"ago",9:"sep",10:"oct",11:"nov",12:"dic"}
DIVISION_IPC_ENERGIA = "alojamiento, agua, electricidad, gas y otros combustibles"


def quitar_tildes(texto):
    texto = str(texto).strip().lower()
    return unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")


def descargar_ipc_dane(anio, mes):
    carpeta = f"{MESES_DANE[mes]}{anio}"
    urls_posibles = [
        f"https://www.dane.gov.co/files/operaciones/IPC/{carpeta}/anex-IPC-{carpeta}.xlsx",  # con subcarpeta
        f"https://www.dane.gov.co/files/operaciones/IPC/anex-IPC-{carpeta}.xlsx",              # sin subcarpeta
    ]
    for url in urls_posibles:
        respuesta = requests.get(url, headers=HEADERS_HTTP, timeout=60)
        if respuesta.status_code == 200:
            return respuesta.content
    raise RuntimeError(f"No se encontró el archivo del DANE para {carpeta}.")


def variacion_anual_ipc_energia(contenido_excel):
    """Busca la división de energía en cualquiera de las hojas del archivo
    y devuelve su variación anual (columna E: Ponderación=B, Var.
    mensual=C, Var. año corrido=D, Var. anual=E)."""
    archivo = pd.ExcelFile(io.BytesIO(contenido_excel))
    for hoja in archivo.sheet_names:
        datos = archivo.parse(hoja, header=None)
        for fila in range(len(datos)):
            celda = datos.iloc[fila, 0]
            if pd.notna(celda) and quitar_tildes(celda) == DIVISION_IPC_ENERGIA:
                return datos.iloc[fila, 4]
    raise RuntimeError("No se encontró la división de energía en el archivo del DANE.")


def extraer_ipc_energia(fecha_inicio, fecha_fin):
    """Descarga el anexo de cada mes del rango solicitado. Si un mes
    puntual falla (por ejemplo, archivo aún no publicado), se omite y se
    sigue con los demás."""
    meses = pd.period_range(pd.to_datetime(fecha_inicio).to_period("M"),
                             pd.to_datetime(fecha_fin).to_period("M"), freq="M")
    filas = []
    for periodo in meses:
        try:
            contenido = solicitar_con_reintentos(lambda: descargar_ipc_dane(periodo.year, periodo.month))
            variacion = variacion_anual_ipc_energia(contenido)
            filas.append({"anio_mes": periodo, "ipc_var_anual_energia": variacion})
        except Exception as e:
            print(f"  IPC energía {periodo}: se omite ({e}).")
        time.sleep(0.3)

    if not filas:
        raise RuntimeError("No se pudo extraer el IPC de energía para ningún mes del rango solicitado.")
    return pd.DataFrame(filas)


## 5. Calendario de festivos (variable auxiliar)

Genera el calendario de festivos, fines de semana y días normales de
Colombia (variable auxiliar, no es una fuente de datos externa).

In [17]:

def extraer_calendario_festivos(anio_inicio, anio_fin):
    try:
        import holidays
    except ImportError:
        import sys, subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "holidays", "--break-system-packages", "-q"])
        import holidays

    festivos_co = holidays.CO(years=range(anio_inicio, anio_fin + 1))
    fechas = pd.date_range(f"{anio_inicio}-01-01", f"{anio_fin}-12-31", freq="D")
    df = pd.DataFrame({"fecha": fechas.date})
    df["es_festivo"] = df["fecha"].isin(festivos_co.keys())
    df["es_fin_de_semana"] = pd.to_datetime(df["fecha"]).dt.dayofweek.isin([5, 6])
    df["tipo_dia"] = np.select(
        [df["es_festivo"], df["es_fin_de_semana"]], ["Festivo", "Fin de semana"], default="Día normal"
    )
    return df[["fecha", "tipo_dia"]]


## 6. Integración — base maestra

Cada fuente se extrae por separado: si una falla, las demás se siguen ejecutando. Llave de integración: **`fecha`**; el IPC, al ser mensual, se propaga a cada día del mes correspondiente.

In [18]:
def construir_base_maestra(fecha_inicio, fecha_fin):
    resultados = {}
    estado = {}

    def extraer(nombre, funcion):
        try:
            resultados[nombre] = funcion()
            estado[nombre] = f"OK ({len(resultados[nombre])} filas)"
        except Exception as e:
            resultados[nombre] = pd.DataFrame()
            estado[nombre] = f"FALLÓ: {e}"

    extraer("demanda", lambda: extraer_metrica_sistema(
        ["Demanda Comercial Total", "Demanda Comercial", "Demanda del SIN", "Demanda"],
        "demanda_gwh", "sum", fecha_inicio, fecha_fin, factor=1 / 1_000_000))
    extraer("precio_bolsa", lambda: extraer_metrica_sistema(
        "Precio de Bolsa Nacional", "precio_bolsa_cop_kwh", "mean", fecha_inicio, fecha_fin))
    extraer("generacion", lambda: extraer_volumen_embalses(fecha_inicio, fecha_fin))
    extraer("trm", lambda: extraer_trm(fecha_inicio, fecha_fin))
    extraer("ipc", lambda: extraer_ipc_energia(fecha_inicio, fecha_fin))
    extraer("festivos", lambda: extraer_calendario_festivos(int(fecha_inicio[:4]), int(fecha_fin[:4])))

    base = pd.DataFrame({"fecha": pd.date_range(fecha_inicio, fecha_fin, freq="D")})

    for nombre in ["demanda", "precio_bolsa", "generacion", "trm"]:
        tabla = resultados[nombre]
        if not tabla.empty:
            tabla = tabla.copy()
            tabla["fecha"] = pd.to_datetime(tabla["fecha"])
            base = base.merge(tabla, on="fecha", how="left")

    if not resultados["festivos"].empty:
        festivos = resultados["festivos"].copy()
        festivos["fecha"] = pd.to_datetime(festivos["fecha"])
        base = base.merge(festivos, on="fecha", how="left")

    if not resultados["ipc"].empty:
        base["anio_mes"] = base["fecha"].dt.to_period("M")
        base = base.merge(resultados["ipc"], on="anio_mes", how="left")
        base = base.drop(columns=["anio_mes"])

    base.to_parquet(f"{RUTA_PROCESADOS}/base_maestra_energia_colombia.parquet", index=False)
    base.to_csv(f"{RUTA_PROCESADOS}/base_maestra_energia_colombia.csv", index=False)
    return base, estado


## 7. Ejecución

Corre el pipeline completo y muestra cuánto tardó y qué se logró extraer.

In [19]:
# Corre el pipeline completo y muestra cuánto tardó y qué se logró extraer.
inicio = time.time()
base_maestra, estado_fuentes = construir_base_maestra(FECHA_INICIO, FECHA_FIN)
print(f"Tiempo total: {round(time.time() - inicio, 1)} s\n")

print("Estado de cada fuente:")
for nombre, mensaje in estado_fuentes.items():
    print(f"  {nombre}: {mensaje}")

base_maestra.head(10)


Catálogo de XM descargado: 193 métricas.
Tiempo total: 7.5 s

Estado de cada fuente:
  demanda: OK (91 filas)
  precio_bolsa: OK (91 filas)
  generacion: OK (91 filas)
  trm: FALLÓ: 0
  ipc: OK (3 filas)
  festivos: OK (366 filas)


,fecha,demanda_gwh,precio_bolsa_cop_kwh,volumen_util_embalses_kwh,tipo_dia,ipc_var_anual_energia
0,2024-01-01,7.344694,219.509494,1.216128e+10,Festivo,9.64
1,2024-01-02,8.612079,398.209740,1.210052e+10,Día normal,9.64
2,2024-01-03,8.952288,459.927215,1.203649e+10,Día normal,9.64
3,2024-01-04,9.141994,551.378046,1.196538e+10,Día normal,9.64
4,2024-01-05,9.220107,625.148652,1.188586e+10,Día normal,9.64
5,2024-01-06,8.843126,627.533787,1.182691e+10,Fin de semana,9.64
6,2024-01-07,8.200566,636.436176,1.177068e+10,Fin de semana,9.64
7,2024-01-08,8.355948,636.694283,1.170797e+10,Festivo,9.64
8,2024-01-09,9.318802,693.135737,1.162915e+10,Día normal,9.64
9,2024-01-10,9.625910,664.416667,1.155686e+10,Día normal,9.64


Validación rápida: tamaño de la base, rango de fechas y valores nulos.

In [20]:
print("Dimensiones:", base_maestra.shape)
print("Rango de fechas:", base_maestra["fecha"].min(), "-", base_maestra["fecha"].max())
print("\nNulos por columna:")
print(base_maestra.isna().sum())


Dimensiones: (91, 6)
Rango de fechas: 2024-01-01 00:00:00 - 2024-03-31 00:00:00

Nulos por columna:
fecha                        0
demanda_gwh                  0
precio_bolsa_cop_kwh         0
volumen_util_embalses_kwh    0
tipo_dia                     0
ipc_var_anual_energia        0
dtype: int64


## 8. Limitaciones

- La disponibilidad hídrica se aproxima con el volumen útil de embalses (agregado nacional); no se clasifica la generación planta por planta.
- El IPC de energía usa la división "Alojamiento, Agua, Electricidad, Gas y Otros Combustibles" del DANE (categoría más cercana disponible, no exclusiva de electricidad); se lee por posición fija de columna, según el formato estándar del anexo del DANE.
- La TRM no se publica de forma idéntica todos los días del calendario, por lo que puede haber valores nulos puntuales.
